# 대학 푸드코트 · 고객 행동과 추천 도입 효과 연구

하나의 시설에 다양한 고객이 여러 날 방문하고, 일부는 지난 경험에 따라 재방문합니다. 같은 평가함수의 PPO를 서로 다른 고객 구성과 시설에서 학습합니다.

미추천, 무작위 추천, 사전 인기 추천, PPO 추천을 비교합니다. 같은 잠재 방문 기회를 주고 실제 방문, 대기, 환불, 재고는 각각 진행합니다. 대시보드는 음수 향상률과 고객 구성별 차이도 표시합니다.

기본은 장기 본 실험 config/adaptive_extended.yaml입니다(목적함수 27개 × 독립 반복 5회, CPU 코어 수만큼 병렬, 하루 이상). 런타임이 끊기면 같은 실험 ID로 다시 실행하면 끝난 작업은 건너뛰고 이어집니다. 더 짧은 본 실험은 config/adaptive_reliable.yaml입니다. 빠른 점검은 config/adaptive_pilot.yaml, 대학 대규모 예시는 config/adaptive_colab.yaml이며, config/adaptive_multitype.yaml은 여러 시설의 용량 부하 시험입니다.

입력 49개와 평가 분모가 바뀌었으므로 새 실험 ID를 사용합니다. Colab이 종료되면 동일 설정으로 다시 실행하면 완료된 운영기간 다음부터 이어집니다.

합성 고객 행동은 현장 관측으로 보정되지 않았습니다. 세부 내용은 저장소 docs/realism_implementation_20261107.md를 확인하세요.


In [ ]:
# 1. 검증된 GitHub 소스 버전 가져오기 및 설치
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/phj9187/foodcourt-adaptive-sim.git'
SOURCE_COMMIT = 'fe04f63662ce092df1d067ee46958721eeb2fc16'  # 642cf41 + kiosk input time
WORK = Path('/content/foodcourt-adaptive-sim')
if not WORK.exists():
    subprocess.run(['git','clone',REPOSITORY,str(WORK)],check=True)
else:
    origin = subprocess.check_output(['git','-C',str(WORK),'remote','get-url','origin'],text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError('같은 경로에 다른 프로젝트가 있습니다. 새 런타임을 사용하세요.')
    subprocess.run(['git','-C',str(WORK),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','--detach',SOURCE_COMMIT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(WORK)],check=True)
# An editable install is only seen by new interpreters; make it importable in this kernel too.
if str(WORK/'src') not in sys.path:
    sys.path.insert(0,str(WORK/'src'))
print('소스 버전:',subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip())
# Multi-runtime runner (scheduling only; does not change the manifest fingerprint)
TOOLS_COMMIT = 'a212c9785bcadf04860b70595d1339a2723477b2'
subprocess.run(['git','-C',str(WORK),'fetch','-q','origin',TOOLS_COMMIT],check=False)
COOP = Path('/content/cooperative_run.py')
COOP.write_text(subprocess.check_output(['git','-C',str(WORK),'show',TOOLS_COMMIT+':tools/cooperative_run.py'],text=True))


In [ ]:
# 2. 핵심 테스트 (모델 학습 실험과는 별개)
test_result=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=WORK,capture_output=True,text=True)
print(test_result.stdout)
print(test_result.stderr)
test_result.check_returncode()
from IPython.display import HTML, display
update_probe=display(HTML('<b>대시보드 갱신 확인 중</b>'),display_id=True)
update_probe.update(HTML('<b>대시보드 화면 갱신 검증 완료</b>'))


In [ ]:
# 3. Google Drive 연결 · 실험 선택
from google.colab import drive
from datetime import datetime, timezone
import re

drive.mount('/content/drive')
CONFIG = 'config/adaptive_extended_input.yaml'  # 약 10~15시간: config/adaptive_reliable.yaml, 빠른 점검: config/adaptive_pilot.yaml
EXPERIMENT = 'extended_input_20261109_001'  # 설정/소스가 달라지면 새 ID를 쓰세요.
if not re.fullmatch(r'[A-Za-z0-9_-]+',EXPERIMENT):
    raise ValueError('실험 ID에는 영문, 숫자, 밑줄, 하이픈만 사용하세요.')
OUTPUT = Path('/content/drive/MyDrive/foodcourt_adaptive_sim')/EXPERIMENT
OUTPUT.mkdir(parents=True,exist_ok=True)
from kiosk_adaptive.study import atomic_json
WORKERS = os.cpu_count() or 1  # 목적함수×반복 작업을 CPU 코어 수만큼 병렬 실행
launch = dict(source_commit=SOURCE_COMMIT,config=CONFIG,experiment=EXPERIMENT)
launch_path = OUTPUT/'launch.json'
if launch_path.exists() and json.loads(launch_path.read_text()) != launch:
    raise RuntimeError('이 폴더는 다른 소스/설정으로 생성됐습니다. 새 실험 ID를 지정하세요.')
atomic_json(launch_path,launch)
print('저장 위치:',OUTPUT)
print('병렬 작업 수:',WORKERS)
print((WORK/CONFIG).read_text())


In [ ]:
# 4. 학습 · 쌍대 비교 실행 / 동일 폴더로 재실행하면 이어서 진행
# 운영기간 하나를 마치고 PPO 갱신할 때마다 체크포인트와 직전 사본 저장.
# 중간에 끊기면 아직 완료하지 못한 운영기간만 다시 계산합니다.
import signal
log_path = OUTPUT/'run.log'
status_path = OUTPUT/'run_status.json'
def status(value,**extra):
    atomic_json(status_path,dict(status=value,updated_utc=datetime.now(timezone.utc).isoformat(),**extra))
command=[sys.executable,'-u','-m','kiosk_adaptive.study','--config',str(WORK/CONFIG),'--output',str(OUTPUT),'--workers',str(WORKERS)]
status('running',command=command)
process=None
from kiosk_adaptive.dashboard import LiveDashboard
live_dashboard=LiveDashboard(OUTPUT,interval=3).start()
try:
    with log_path.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            log.write(line)
        code=process.wait()
    status('complete' if code==0 else 'failed',exit_code=code)
    if code:raise RuntimeError(f'실행 실패: {code}. run.log를 확인하세요.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try:process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill();process.wait()
    status('interrupted')
    print('중단됨. 같은 설정으로 다시 실행하면 마지막 저장 지점부터 이어갑니다.')
except Exception as exc:
    status('failed',error=str(exc))
    raise

finally:
    live_dashboard.stop()


In [ ]:
# 4-B. 여러 런타임 동시 실행 (4번 대신 실행)
# 여러 Colab 런타임이 같은 실험 폴더에서 작업을 나눠 맡습니다. 작업마다 claim.json을 남겨 중복 학습을 막고,
# 모든 목적함수의 반복 0~2를 먼저 끝낸 뒤 반복 3~4를 진행합니다. 결과는 4번과 동일합니다(작업별 시드 고정).
# 실험 폴더와 같은 torch 빌드 사용 (manifest 일치; GPU/TPU 런타임은 기본 torch가 다름)
_v=subprocess.run([sys.executable,'-c','import torch,numpy;print(torch.__version__,numpy.__version__)'],capture_output=True,text=True).stdout.strip()
print('torch/numpy:',_v)
if not _v.startswith('2.14.1+cpu'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','torch==2.14.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    print('torch/numpy:',subprocess.run([sys.executable,'-c','import torch,numpy;print(torch.__version__,numpy.__version__)'],capture_output=True,text=True).stdout.strip())
import socket
RUNTIME = socket.gethostname()
coop_log = OUTPUT/f'run_{RUNTIME}.log'
coop_status = OUTPUT/f'run_status_{RUNTIME}.json'
def coop_state(value,**extra):
    atomic_json(coop_status,dict(status=value,runtime=RUNTIME,workers=WORKERS,updated_utc=datetime.now(timezone.utc).isoformat(),**extra))
command=[sys.executable,'-u',str(COOP),'--config',str(WORK/CONFIG),'--output',str(OUTPUT),'--workers',str(WORKERS),'--runtime',RUNTIME]
coop_state('running',command=command)
process=None
try:
    with coop_log.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            log.write(line)
            if ' start ' in line or ' done ' in line or 'FAILED' in line or 'aggregated' in line or 'runtime ' in line:
                print(line,end='')
        code=process.wait()
    coop_state('complete' if code==0 else 'failed',exit_code=code)
    if code:raise RuntimeError(f'실행 실패: {code}. {coop_log.name}를 확인하세요.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try:process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill();process.wait()
    coop_state('interrupted')
    print('중단됨. 다시 실행하면 마지막 체크포인트부터 이어갑니다.')


In [ ]:
# 5. 결과 확인 — 함수별 자기 기준의 AI 도입 효과
import csv
path=OUTPUT/'comparison.csv'
if path.exists():
    with path.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    for row in rows:
        print(row['id'], '반복',row['replica'],row['scenario'],
              '전체점수 변화',round(float(row['mean_delta']),6),
              '고객점수 변화',round(float(row['mean_customer_delta']),6),
              '이익 변화(원)',round(float(row['mean_profit_delta_won']),1))
else:
    print('아직 전체 실험이 끝나지 않았습니다. 모델별 progress.json과 summary.json을 확인하세요.')
